# 14g — Pull V-Dem ERT (Episodes of Regime Transformation)

**Source:** Edgell, Lührmann, Maerz et al. — *Episodes of Regime Transformation* dataset  
**Provider:** V-Dem Institute, University of Gothenburg  
**Coverage:** ~180 countries, 1900–present, annual  
**Access:** Direct CSV download from V-Dem website (no registration required)

## What this notebook does

1. Downloads the V-Dem ERT dataset CSV from the V-Dem website
2. Selects autocratization-episode variables used in the backsliding literature
3. Maps V-Dem country identifiers to ISO3
4. Writes a country-year parquet to ADLS

## Why ERT is needed

Notebook 03 pulls V-Dem core democracy indices but **does not pull ERT**.
ERT is a separate dataset that codes discrete *episodes* of regime change —
when a country enters or exits an autocratization or democratization episode —
rather than just the level of a democracy index at a point in time.

The synthesis document identifies the following ERT variables as predictors
that appear in multiple instability models but are absent from current acquisition:

- `aut_ep` — country is in an autocratization episode this year
- `aut_ep_start` — year the current autocratization episode started
- `dem_ep` — country is in a democratization episode this year
- `reg_trans_type` — episode type (autocratization vs. democratization)

These differ from the continuous V-Dem indices already in nb03: a country can have
a moderate `v2x_libdem` score while not being in an active episode, or it can be
coded as autocratizing even if the score hasn't yet crossed a threshold. The episode
coding captures directional movement and trajectory, not just level.

## ERT episode logic

An **autocratization episode** begins when the Electoral Democracy Index (EDI)
falls significantly relative to its recent peak, continues while the decline
persists, and ends when the country either stabilises (regime survives the
episode) or reaches a fully closed autocracy. The ERT codebook distinguishes:

| Type | Description |
|---|---|
| Gradual autocratization | Slow EDI decline, often through institutional erosion |
| Rapid autocratization | Sharp EDI decline, often coup-driven |
| Gradual democratization | Slow EDI rise |
| Rapid democratization | Sharp EDI rise |

## Variables produced

| Variable | Description | Use |
|---|---|---|
| `aut_ep` | 1 if country is in an autocratization episode | **Dependent variable** |
| `aut_ep_start_year` | Year the current episode started (NA if no episode) | Predictor (episode duration) |
| `aut_ep_duration` | Years elapsed since episode start | Predictor |
| `dem_ep` | 1 if country is in a democratization episode | Predictor |
| `reg_trans_type` | Episode type string | Diagnostic |
| `edi_change_3y` | 3-year change in Electoral Democracy Index | Predictor (trend) |

## Output path on ADLS

```
raw/vdem_ert/{RUN_DATE}/vdem_ert_annual.parquet
```

Add `"vdem_ert": "raw/vdem_ert"` to `RAW_PREFIXES` in notebook 14.

In [ ]:
%%capture
%pip install requests pandas pyarrow azure-identity adlfs --quiet

In [ ]:
import io
import os
import warnings
from datetime import datetime
from pathlib import Path

import pandas as pd
import numpy as np
import requests
import adlfs
from azure.identity import DefaultAzureCredential

warnings.filterwarnings('ignore')
RUN_DATE = datetime.utcnow().strftime('%Y%m%d')

In [ ]:
ADLS_ACCOUNT_NAME = os.environ['ADLS_ACCOUNT_NAME']
ADLS_CONTAINER    = os.getenv('ADLS_CONTAINER', 'data')
CROSSWALK_PATH    = '../data/country_crosswalk.csv'

# V-Dem ERT dataset — direct CSV download from V-Dem website.
# Version numbers increment with annual releases; try recent versions in order.
# Check https://www.v-dem.net/data/dataset-archive/ for current version.
ERT_URLS = [
    'https://v-dem.net/media/datasets/ERT_Country_Year_V14.csv',
    'https://v-dem.net/media/datasets/ERT_Country_Year_V13.csv',
    'https://v-dem.net/media/datasets/ERT_Country_Year_V12.csv',
]

# ERT column names (stable across versions — uses the V-Dem variable naming convention)
# Primary episode flag columns
ERT_AUT_COL   = 'reg_type'         # regime transformation type per country-year
ERT_EP_COLS = [
    'aut_ep',           # in autocratization episode (0/1)
    'aut_ep_start_year',# year episode began
    'aut_ep_end_year',  # year episode ended (NA if ongoing)
    'aut_ep_outcome',   # outcome code
    'dem_ep',           # in democratization episode (0/1)
    'dem_ep_start_year',
    'dem_ep_end_year',
    'reg_trans_type',   # 'gradual_aut', 'rapid_aut', 'gradual_dem', 'rapid_dem'
]
ERT_EDI_COL   = 'v2x_polyarchy'    # Electoral Democracy Index (EDI) in ERT file

In [ ]:
credential = DefaultAzureCredential()
storage_options = {'account_name': ADLS_ACCOUNT_NAME, 'credential': credential}

def adls_path(subpath):
    return f'abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}.dfs.core.windows.net/{subpath}'

def write_parquet(df, subpath):
    df.to_parquet(adls_path(subpath), storage_options=storage_options,
                  index=False, engine='pyarrow')
    print(f'  Written {len(df):,} rows → {subpath}')

In [ ]:
df_raw = None
for url in ERT_URLS:
    try:
        print(f'Downloading V-Dem ERT from {url} ...')
        resp = requests.get(url, timeout=120,
                            headers={'User-Agent': 'Mozilla/5.0'})
        resp.raise_for_status()
        df_raw = pd.read_csv(io.BytesIO(resp.content), low_memory=False)
        print(f'  Raw shape: {df_raw.shape}')
        print(f'  Columns  : {df_raw.columns.tolist()[:30]}')
        break
    except Exception as exc:
        print(f'  Failed: {exc}')

if df_raw is None:
    raise RuntimeError(
        'Could not download V-Dem ERT data. '
        'Download manually from https://www.v-dem.net/data/dataset-archive/ '
        '(search for ERT Country-Year) and load with: '
        'df_raw = pd.read_csv("ERT_Country_Year_V14.csv")'
    )

In [ ]:
# Normalise column names
df = df_raw.copy()
df.columns = [str(c).strip().lower() for c in df.columns]
print('Columns:', df.columns.tolist())

# Identify country identifier column — ERT uses V-Dem's own country_text_id (ISO3-like)
# or country_name depending on version
id_candidates = ['country_text_id', 'iso3', 'countrycode', 'country_id']
name_candidates = ['country_name', 'country']

iso_col  = next((c for c in id_candidates if c in df.columns), None)
name_col = next((c for c in name_candidates if c in df.columns), None)
year_col = next((c for c in ['year', 'Year'] if c in df.columns), 'year')

print(f'ISO column: {iso_col}, Name column: {name_col}, Year column: {year_col}')
print(f'Years: {df[year_col].min()}–{df[year_col].max()}')

# ERT episode columns present in this version
ert_cols_present = [c for c in ERT_EP_COLS if c in df.columns]
print(f'ERT episode columns present: {ert_cols_present}')

In [ ]:
# Map to ISO3
# V-Dem's country_text_id is a 3-character code that is close to ISO3 but has
# a handful of differences for historical states. Use it directly where it
# matches, then fall back to name lookup via crosswalk.
cw_path = Path(CROSSWALK_PATH)
if not cw_path.exists():
    cw_path = Path('data/country_crosswalk.csv')

df_cw = pd.read_csv(cw_path, dtype=str)
name_to_iso3 = dict(zip(df_cw['country_name_canonical'].str.lower(), df_cw['iso3']))

# V-Dem country_text_id deviations from ISO3
vdem_id_overrides = {
    'ZZB': 'ZZB',  # Zanzibar — no ISO3, drop downstream
    'PSG': 'PSE',  # Palestine/Gaza
    'SML': 'SOM',  # Somaliland (not a sovereign state — merge to Somalia)
    'COK': 'COK',  # Cook Islands
    'XKX': 'XKX',  # Kosovo
}

vdem_name_overrides = {
    'united states of america': 'USA',
    'united states':            'USA',
    'russia':                   'RUS',
    'south korea':              'KOR',
    'north korea':              'PRK',
    'iran':                     'IRN',
    'syria':                    'SYR',
    'vietnam':                  'VNM',
    "democratic republic of the congo": 'COD',
    "republic of the congo":    'COG',
    "ivory coast":              'CIV',
    "cote d'ivoire":            'CIV',
    "cape verde":               'CPV',
    "east timor":               'TLS',
    "timor-leste":              'TLS',
    "eswatini":                 'SWZ',
    "north macedonia":          'MKD',
}

def _vdem_to_iso3(row):
    if iso_col and pd.notna(row.get(iso_col)):
        code = str(row[iso_col]).strip().upper()
        return vdem_id_overrides.get(code, code)
    if name_col and pd.notna(row.get(name_col)):
        n = str(row[name_col]).strip().lower()
        return vdem_name_overrides.get(n) or name_to_iso3.get(n)
    return None

df['iso3'] = df.apply(_vdem_to_iso3, axis=1)

unmatched_iso3s = df[df['iso3'].isna()][iso_col if iso_col else name_col].unique()
print(f'Unmatched ({len(unmatched_iso3s)}): {sorted(str(u) for u in unmatched_iso3s)[:15]}')

df = df.dropna(subset=['iso3'])
df['year'] = pd.to_numeric(df[year_col], errors='coerce').astype(int)

In [ ]:
# Select and clean ERT columns

# Core episode flags — present in all ERT versions
for col in ['aut_ep', 'dem_ep']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)
    else:
        df[col] = 0

# Episode start year — used to compute duration
for col in ['aut_ep_start_year', 'dem_ep_start_year']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    else:
        df[col] = np.nan

# Episode duration: years elapsed since episode start
df['aut_ep_duration'] = np.where(
    df['aut_ep'] == 1,
    df['year'] - df['aut_ep_start_year'],
    0,
)

# Episode outcome (if present)
if 'aut_ep_outcome' in df.columns:
    df['aut_ep_outcome'] = df['aut_ep_outcome'].astype(str).str.strip()

# Transformation type string
if 'reg_trans_type' not in df.columns:
    # Derive from episode flags if the column is absent
    df['reg_trans_type'] = np.select(
        [df['aut_ep'] == 1, df['dem_ep'] == 1],
        ['autocratization', 'democratization'],
        default='none',
    )

# EDI (Electoral Democracy Index) — included in ERT file; use to compute trend
edi_col = next((c for c in [ERT_EDI_COL, 'edi', 'v2x_polyarchy'] if c in df.columns), None)
if edi_col:
    df = df.sort_values(['iso3', 'year'])
    df['edi'] = pd.to_numeric(df[edi_col], errors='coerce')
    df['edi_change_3y'] = df.groupby('iso3')['edi'].transform(
        lambda x: x - x.shift(3)
    )
else:
    df['edi_change_3y'] = np.nan

print(f'Rows: {len(df):,}')
print(f'Countries: {df["iso3"].nunique()}')
print(f'aut_ep base rate: {df["aut_ep"].mean():.2%}')
print(f'dem_ep base rate: {df["dem_ep"].mean():.2%}')

In [ ]:
# Assemble final panel
final_cols = [c for c in [
    'iso3', 'year',
    'aut_ep', 'aut_ep_start_year', 'aut_ep_duration', 'aut_ep_outcome',
    'dem_ep', 'dem_ep_start_year',
    'reg_trans_type',
    'edi', 'edi_change_3y',
] if c in df.columns]

df = df[final_cols].sort_values(['iso3', 'year']).reset_index(drop=True)

print(f'Final panel: {len(df):,} country-years, {df["iso3"].nunique()} countries')
print(f'Years: {df["year"].min()}–{df["year"].max()}')

if 'reg_trans_type' in df.columns:
    print('Episode type distribution:')
    print(df.loc[df['aut_ep']==1, 'reg_trans_type'].value_counts().to_string())

df.head(10)

In [ ]:
write_parquet(df, f'raw/vdem_ert/{RUN_DATE}/vdem_ert_annual.parquet')

print()
print('=' * 55)
print('Notebook 14g complete')
print('=' * 55)
print(f'  Rows      : {len(df):,}')
print(f'  Countries : {df["iso3"].nunique()}')
print(f'  Years     : {df["year"].min()}–{df["year"].max()}')
print()
print('Dependent variable written:')
print('  aut_ep          — 1 if country is in an autocratization episode')
print()
print('Predictors written:')
print('  aut_ep_start_year, aut_ep_duration — episode history')
print('  dem_ep, dem_ep_start_year          — democratization episodes')
print('  reg_trans_type                     — episode type string')
print('  edi, edi_change_3y                 — Electoral Democracy Index trend')
print()
print('Note: nb03 pulls V-Dem core indices (libdem, polyarchy, etc.).')
print('This notebook adds only the ERT episode variables not present in nb03.')
print()
print('Next: add  "vdem_ert": "raw/vdem_ert"  to RAW_PREFIXES in notebook 14.')